# 1.2 订单清洗

按 1.1 数出来的问题逐项处理：完全重复行和未来日期删掉；商品名称统一写法；
退货行和金额未知只做标记，不删——它们是真实发生的业务，删了账就对不上。

In [1]:
import sys
import unicodedata
from pathlib import Path

import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from demo_lib import LAST_MONTH, out

OUT = out("1.2")
run = dsflow.start_run("1.2", project=ROOT, hypothesis="删除完全重复行和未来日期后，其余订单行全部保留，数量与金额不被改动")

run.log_input(ROOT / "data/raw/orders.csv", name="orders_raw", stage="raw")
raw = pd.read_csv(ROOT / "data/raw/orders.csv", dtype={"订单号": str, "SKU": str})
print(f"清洗前 {len(raw):,} 行")


清洗前 121,000 行


In [2]:
dedup = raw.drop_duplicates()
placed = pd.to_datetime(dedup["下单时间"])
future = placed > f"{LAST_MONTH}-30 23:59:59"
clean = dedup[~future].copy()
ledger = pd.DataFrame({
    "处理": ["原始", "删除完全重复行", "删除未来日期", "清洗后"],
    "行数": [len(raw), len(raw) - len(dedup), int(future.sum()), len(clean)],
})
ledger.to_csv(OUT / "row_ledger.csv", index=False)
ledger


,处理,行数
0,原始,121000
1,删除完全重复行,1000
2,删除未来日期,2
3,清洗后,119998


In [3]:
before = clean["商品名称"]
clean["商品名称"] = before.map(lambda x: unicodedata.normalize("NFKC", x).strip())
clean["下单月份"] = placed[~future].dt.strftime("%Y-%m")
clean["是否退货"] = clean["数量"] < 0
clean["金额未知"] = clean["金额"].isna()
clean = clean.reset_index(drop=True)
metrics = {
    "删除完全重复行": len(raw) - len(dedup),
    "删除未来日期": int(future.sum()),
    "商品名称规范化行": int((clean["商品名称"] != before.reset_index(drop=True)).sum()),
    "退货行": int(clean["是否退货"].sum()),
    "金额未知行": int(clean["金额未知"].sum()),
    "清洗后订单行": len(clean),
}
for k, v in metrics.items():
    print(f"{k}：{v:,}")


删除完全重复行：1,000
删除未来日期：2
商品名称规范化行：3,703
退货行：361
金额未知行：636
清洗后订单行：119,998


In [4]:
run.log_output(clean, name="orders_clean", path=OUT / "orders_clean.parquet", stage="processed",
               description="清洗后订单明细，一行 = 一个订单行",
               replaces=["orders_raw"])  # 清洗后的表替代原始表：后面的步骤都用它，原始表退出后存量
run.log_metrics(metrics)
run.log_artifact(OUT / "row_ledger.csv", purpose="行数台账：每一步删了多少行", kind="table")
conclusion = (
    f"订单行 {len(raw):,} → {len(clean):,}：删除完全重复行 {metrics['删除完全重复行']:,}、未来日期 {metrics['删除未来日期']}；"
    f"退货行 {metrics['退货行']:,}、金额未知行 {metrics['金额未知行']:,} 只标记不删除"
)
run.set_conclusion(conclusion, validity="有效")
run.end()
print(conclusion)


订单行 121,000 → 119,998：删除完全重复行 1,000、未来日期 2；退货行 361、金额未知行 636 只标记不删除
